# 02 - Split Data dan EDA Singkat

Input: `daily_kurs.csv` dan `articles_aligned.csv` dari `data/processed/` (hasil notebook 01).

Output di `data/split/`: `daily_kurs_split.csv`, `articles_split.csv` (keduanya punya kolom `split`), dan `split_info.json`.

Split dilakukan sebelum ekstraksi fitur karena TF-IDF harus di-fit hanya pada data train. Kalau vocabulary dihitung dari seluruh data, informasi test ikut masuk ke fitur (data leakage).

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

def find_repo_root():
    """Cari folder root repo (yang berisi data/ dan src/), dari mana pun notebook dijalankan."""
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data").is_dir() and (d / "src").is_dir():
            return d
    raise FileNotFoundError(f"Root repo (berisi data/ dan src/) tidak ditemukan dari {here}")

ROOT = find_repo_root()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
SPLIT_DIR = ROOT / "data" / "split"
for d in (PROCESSED_DIR, SPLIT_DIR):
    d.mkdir(parents=True, exist_ok=True)

TRAIN_FRAC, VAL_FRAC = 0.70, 0.15     # sisanya test

MIN_MONTHLY_COVERAGE = 0.5      # bulan dianggap "tercakup" kalau >= 50% hari bursanya punya berita
TRIM_TO_COVERED_PERIOD = True   # potong ujung awal/akhir yang belum tercakup berita

daily = pd.read_csv(PROCESSED_DIR / "daily_kurs.csv", parse_dates=["date"])
arts = pd.read_csv(PROCESSED_DIR / "articles_aligned.csv", parse_dates=["trading_date"])
print(daily.shape, arts.shape)

## 0. Cek cakupan berita

Data kurs JISDOR penuh, tapi hasil scraping berita belum tentu merata di seluruh periode. Sebelum split, dilihat berapa persen hari bursa per bulan yang punya minimal satu artikel. Bulan dengan cakupan rendah ditandai merah.

Kalau ujung awal atau akhir periode belum tercakup (misalnya scraping baru mulai dari 2023), periode dipotong ke rentang bulan-bulan yang tercakup, supaya model tidak belajar dari ribuan hari "tanpa berita" yang sebenarnya hanya belum ter-scrape. Celah di tengah periode tidak dipotong; hari-harinya tetap ada dan diberi penanda `no_news` di notebook 03.

Ini solusi sementara. Tugas 1 meminta berita 5 tahun, jadi cara terbaik tetap melengkapi scraping di bulan yang bolong, sebaiknya dengan jumlah per bulan yang kurang lebih sama.

In [ ]:
has_news = daily["date"].isin(arts["trading_date"].unique())
cov = daily.assign(has_news=has_news).set_index("date")["has_news"].resample("MS").mean()
covered = cov[cov >= MIN_MONTHLY_COVERAGE]

print(f"Bulan tercakup: {len(covered)} dari {len(cov)}")
low = [d.strftime("%Y-%m") for d in cov[cov < MIN_MONTHLY_COVERAGE].index]
print("Bulan dengan cakupan rendah:", low if low else "tidak ada")

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(cov.index, cov.values, width=25,
       color=["#4c78a8" if v >= MIN_MONTHLY_COVERAGE else "#d9534f" for v in cov.values])
ax.axhline(MIN_MONTHLY_COVERAGE, color="gray", ls="--", lw=1)
ax.set_ylim(0, 1)
ax.set_title("Porsi hari bursa yang punya berita, per bulan")
plt.tight_layout()
plt.show()

if TRIM_TO_COVERED_PERIOD and len(covered):
    start_m = covered.index.min()
    end_m = covered.index.max() + pd.offsets.MonthEnd(0)
    n_days_before, n_arts_before = len(daily), len(arts)
    daily = daily[(daily["date"] >= start_m) & (daily["date"] <= end_m)].reset_index(drop=True)
    arts = arts[(arts["trading_date"] >= start_m) & (arts["trading_date"] <= end_m)].copy()
    print(f"Periode dipakai: {start_m.date()} -> {end_m.date()} "
          f"(hari bursa {n_days_before} -> {len(daily)}, artikel {n_arts_before} -> {len(arts)})")

## 1. Split kronologis

Data dipotong menurut waktu: train paling awal, lalu validation, lalu test paling akhir, tanpa diacak. Data time-series yang diacak membuat model "melihat masa depan" saat training. Split dilakukan di level hari bursa; artikel mengikuti split dari `trading_date`-nya, jadi tidak ada satu hari yang terbelah ke dua split.

In [ ]:
daily = daily.sort_values("date").reset_index(drop=True)
n = len(daily)
i_val = int(n * TRAIN_FRAC)
i_test = int(n * (TRAIN_FRAC + VAL_FRAC))

daily["split"] = "train"
daily.loc[daily.index >= i_val, "split"] = "val"
daily.loc[daily.index >= i_test, "split"] = "test"

arts = arts.merge(
    daily[["date", "split"]].rename(columns={"date": "trading_date"}),
    on="trading_date", how="inner",
)

info = {}
for s in ["train", "val", "test"]:
    d = daily[daily["split"] == s]
    info[s] = {"start": str(d["date"].min().date()), "end": str(d["date"].max().date()),
               "n_days": int(len(d)), "n_articles": int((arts["split"] == s).sum()),
               "share_up": round(float(d["naik_besok"].mean()), 3)}
info_df = pd.DataFrame(info).T
print(info_df)

json.dump(info, open(SPLIT_DIR / "split_info.json", "w"), indent=2)
daily.to_csv(SPLIT_DIR / "daily_kurs_split.csv", index=False)
arts.to_csv(SPLIT_DIR / "articles_split.csv", index=False)
print("Tersimpan di:", SPLIT_DIR)

Kolom `share_up` menunjukkan proporsi hari kurs naik. Kalau angkanya dekat 0.5, model yang menebak acak sudah mendapat akurasi sekitar 50%. Kalau timpang, misalnya 0.6, model yang selalu menebak "naik" sudah 60%, dan itu batas bawah yang harus dilampaui.

## 2. EDA: kurs

Grafik pertama menampilkan seluruh periode dengan batas split. Statistik dan histogram sisanya dihitung dari data train saja, supaya keputusan modeling tidak dipengaruhi oleh data test.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))

colors = {"train": "#cfe3f5", "val": "#fde9c0", "test": "#f6cfd0"}
ax[0].plot(daily["date"], daily["kurs_jisdor"], lw=1, color="#1f3b5c")
for s, g in daily.groupby("split"):
    ax[0].axvspan(g["date"].min(), g["date"].max(), color=colors[s], alpha=0.6, label=s)
ax[0].set_title("JISDOR dan pembagian split")
ax[0].legend(loc="upper left")

train = daily[daily["split"] == "train"]
ax[1].hist(train["log_return"], bins=50, color="#4c78a8")
ax[1].set_title("Sebaran log-return harian (train)")

share = daily.groupby("split")["naik_besok"].mean().reindex(["train", "val", "test"])
ax[2].bar(share.index, share.values, color="#4c78a8")
ax[2].axhline(0.5, color="gray", ls="--", lw=1)
ax[2].set_ylim(0, 1)
ax[2].set_title("Proporsi hari kurs naik")

plt.tight_layout()
plt.show()

print("Log-return train: mean {:.5f}, std {:.5f}, skew {:.2f}".format(
    train["log_return"].mean(), train["log_return"].std(), train["log_return"].skew()))

## 3. Baseline persistence dan rentang kurs

Baseline persistence memprediksi `kurs_besok` = `kurs_jisdor` hari ini. RMSE dan MAE-nya di tiap split adalah angka yang harus dikalahkan model mana pun.

Tabel ini juga menampilkan rentang kurs per split. Ini penting kalau nanti memakai model berbasis pohon (XGBoost, random forest): model itu tidak bisa memprediksi di luar rentang nilai yang pernah dilihatnya saat training. Kalau kurs di val/test melewati maksimum train, prediksinya mentok di batas itu dan errornya membesar. Kalau banyak hari yang keluar rentang, latih model pada `delta_besok`, lalu prediksi levelnya = `kurs_jisdor` + prediksi selisih. Evaluasinya tetap pada level kurs, jadi tetap sesuai definisi target dan sebanding dengan baseline.

In [ ]:
rows = []
for s in ["train", "val", "test"]:
    d = daily[daily["split"] == s]
    err = d["kurs_besok"] - d["kurs_jisdor"]          # error persistence = delta_besok
    rows.append({"split": s,
                 "RMSE_persistence": np.sqrt((err ** 2).mean()),
                 "MAE_persistence": err.abs().mean(),
                 "kurs_min": d["kurs_jisdor"].min(),
                 "kurs_max": d["kurs_jisdor"].max()})
print(pd.DataFrame(rows).set_index("split").round(2))

tr = daily[daily["split"] == "train"]
lo, hi = tr["kurs_besok"].min(), tr["kurs_besok"].max()
other = daily[daily["split"] != "train"]
outside = ((other["kurs_besok"] > hi) | (other["kurs_besok"] < lo)).sum()
print(f"\nHari val/test dengan kurs_besok di luar rentang train ({lo:,.0f}-{hi:,.0f}): {outside} dari {len(other)}")

## 4. EDA: berita

Yang dicek: berapa artikel per hari bursa, seberapa banyak hari tanpa berita, tren jumlah artikel per bulan, panjang artikel, dan seberapa banyak berita yang menumpuk dari akhir pekan atau libur (`gap_days`). Hari tanpa berita menentukan cara pengisian nilai kosong di notebook 03.

In [ ]:
train_arts = arts[arts["split"] == "train"]
per_day = arts.groupby("trading_date").size().reindex(daily["date"]).fillna(0).astype(int)
daily["n_articles"] = per_day.values

fig, ax = plt.subplots(2, 2, figsize=(14, 7))

tr = daily[daily["split"] == "train"]
ax[0, 0].hist(tr["n_articles"], bins=range(0, int(tr["n_articles"].max()) + 2), color="#4c78a8")
ax[0, 0].set_title("Artikel per hari bursa (train)")

monthly = arts.set_index("trading_date").resample("MS").size()
ax[0, 1].plot(monthly.index, monthly.values, marker="o", ms=3, color="#1f3b5c")
ax[0, 1].set_title("Artikel per bulan (semua split)")

ax[1, 0].hist(train_arts["n_words"], bins=50, color="#4c78a8")
ax[1, 0].set_title("Panjang artikel dalam kata (train)")

gap = train_arts["gap_days"].value_counts().sort_index()
ax[1, 1].bar(gap.index.astype(str), gap.values, color="#4c78a8")
ax[1, 1].set_title("gap_days (0 = terbit di hari bursa yang sama)")

plt.tight_layout()
plt.show()

print("Hari tanpa berita per split:")
print((daily.assign(no_news=daily["n_articles"] == 0).groupby("split")["no_news"].mean().round(3)))

## 5. Cek awal: apakah jumlah berita berhubungan dengan gejolak kurs?

Pengecekan sederhana pada data train: korelasi antara jumlah artikel di hari itu dan besarnya pergerakan kurs (nilai absolut return). Bukan pembuktian hipotesis, hanya gambaran awal apakah hari yang ramai berita cenderung lebih bergejolak.

In [ ]:
tr = daily[daily["split"] == "train"]
rho = tr["n_articles"].corr(tr["log_return"].abs(), method="spearman")
print(f"Spearman(n_articles, |log_return|) di train: {rho:.3f}")